# (final) All mapped concepts: evaluation and analysis

Benchmark on the **mapped concepts $M$** (SNOMED CT concepts mapped at HUG) with the full candidate subgraph $G_D$. This notebook merges notebooks 4 (evaluation) and 5 (analysis) and replaces them.

**Metrics** (`src/graph_tokenizer_gd_tree_dev/eval.py`)
* **Core, used for the ranking and the figures:** `br_weighted_PC` (branch-weighted PC, ↑), `unique_rate` (↑), `unknown_rate` (↓), `tokens_per_concept` (↓).
* `br_weighted_PC` uses the same expansion as the context trees. Every concept splits its weight equally between its outgoing relationships, so many tokens in one branch can no longer hide a branch that ends without a token.
* **Informational:** `num_effective_token`, the number of selected tokens that at least one concept uses, and its share of $k$. It is not ranked.
* **Versioning only:** `distance_score` (the current PC) and `semantic_coverage` are still computed and saved, but not used below.

| Section | Paper |
|---|---|
| 1 Data and candidate subgraph | experimental setup; appendix (absent relation types) |
| 2 Evaluation | metrics; the 19 compared configurations |
| 3 Ranking | benchmark: ranking figure; appendix ranking table |
| 4 Effect of $k$ | benchmark: metrics vs $k$; appendix figure with random |
| 5 Effect of $\lambda$ | appendix: $\lambda$ table and figure |
| 6 Per-concept distribution | benchmark: distribution at fixed $k$ |
| 7 Effective tokens | informational |
| 8 Metric correlations | discussion |
| 9 Key numbers | numbers quoted in the text |

**Run** from the repository root, top to bottom. Evaluations are cached in `OUT_DIR`, one file per group of methods. A file is recomputed only if it is missing, lacks the new metrics, or `RECOMPUTE = True`. A full evaluation takes about as long as notebook 4 (about 2 hours). Results go to `OUT_DIR` and figures to `figures/all_mapped/`, so the files of notebooks 4 and 5 are not overwritten.

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import glob
import json
import multiprocessing as mp
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import polars as pl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MultipleLocator
from tqdm.auto import tqdm

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.eval as ev
import src.graph_tokenizer_gd_tree_dev.fidelity_eval as fe
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.utils as utils

## 0. Configuration

Every choice of this notebook lives in this cell.

In [ ]:
# --- what to run -----------------------------------------------------------------------------------
RECOMPUTE = False          # True: rerun every evaluation, even when its result file exists
SHOW_GRAPH_STATS = True    # Section 1 rebuilds the full ontology graph to compare it with G_D (slow)
N_WORKERS = os.cpu_count()

# --- benchmark ---------------------------------------------------------------------------------------
D = config.TokenizerParam().max_dist_candidate                    # search horizon (9)
KS = sorted(int(k) for k in config.TokenizerParam().Ks)           # 39 budgets: 500, 1,000, ..., 19,500
RND_ITERS = [int(i) for i in config.TokenizerParam().rnd_iters]   # 20 random draws per budget
LAM_MAIN = "0.9"                                                  # lambda used in the rest of the paper
KS_DIST = [3500, 10000, 16500]                                    # one budget per compression level
LAMS_DIST = ["0.3", "0.6", "0.9", "1.0"]                          # lambdas of the distribution figure

# --- outputs: new folders, so the results and figures of notebooks 4/5 stay untouched -------------
OUT_DIR = os.path.join(config.Results().path, "all_mapped_v2") + "/"
FIG_DIR = "figures/all_mapped/"
for p in (OUT_DIR, FIG_DIR):
    os.makedirs(p, exist_ok=True)

# --- methods ------------------------------------------------------------------------------------------
OURS_FILES = utils.to_type_dict(glob.glob(os.path.join(config.CandidateLists().path_greedy_tree, "*.parquet")))
LAMBDA_IDS = sorted(OURS_FILES, key=float)                        # every lambda run, incl. the fine grid
LAMBDA_COARSE = [l for l in LAMBDA_IDS if abs(10 * float(l) - round(10 * float(l))) < 1e-9]   # 0.0, ..., 1.0
ABLATION = "discrete_set_cover"
BASELINES = ["highest_degree", "highest_degree_dist_1", "most_children",
             "pagerank", "personalized_pagerank", "closeness_centrality"]
RANDOM = "k_random_all_samples"
RANKED_METHODS = LAMBDA_COARSE + [ABLATION] + BASELINES + [RANDOM]   # the compared configurations

FAMILY = {**{l: "ours" for l in LAMBDA_IDS}, ABLATION: "ours(ablation)",
          "highest_degree": "degree-based", "highest_degree_dist_1": "degree-based", "most_children": "degree-based",
          "pagerank": "centrality-based", "personalized_pagerank": "centrality-based",
          "closeness_centrality": "centrality-based", RANDOM: "random"}
DISPLAY = {**{l: f"Ours ($\\lambda$ = {l})" for l in LAMBDA_IDS}, ABLATION: "Set cover (ablation)",
           "highest_degree": "Highest degree ($d \\leq D$)", "highest_degree_dist_1": "Highest degree ($d \\leq 1$)",
           "most_children": "Most IS\\_A children", "pagerank": "PageRank",
           "personalized_pagerank": "Personalized PageRank", "closeness_centrality": "Closeness centrality",
           RANDOM: "Random"}

# --- metrics -------------------------------------------------------------------------------------------
CORE = ["br_weighted_PC", "unique_rate", "unknown_rate", "tokens_per_concept"]   # ranking and figures
HIB = {"br_weighted_PC": True, "unique_rate": True, "unknown_rate": False, "tokens_per_concept": False,
       "uncovered_branch_rate": False, "num_effective_token": True, "effective_token_share": True}
METRIC_TITLE = {
    "br_weighted_PC": "Branch-weighted PC ($\\uparrow$)",
    "unique_rate": "Unique rate ($\\uparrow$)",
    "unknown_rate": "Unknown rate ($\\downarrow$)",
    "tokens_per_concept": "Tokens per concept ($\\downarrow$)",
    "uncovered_branch_rate": "Uncovered branch rate ($\\downarrow$)",
    "effective_token_share": "Effective tokens / $k$ ($\\uparrow$)",
}
# names of eval.evaluate -> names used here (notebook 4 renamed them the same way)
RENAME = {"conciseness": "tokens_per_concept", "unk_rate": "uncovered_branch_rate", "uncovered_rate": "unknown_rate"}

# --- compression levels: three equal groups of budgets ----------------------------------------------
n_lvl = len(KS) // 3
LEVELS = {"high": (KS[0], KS[n_lvl - 1]), "mid": (KS[n_lvl], KS[2 * n_lvl - 1]), "low": (KS[2 * n_lvl], KS[-1])}


def level_expr():
    e = pl.when(pl.col("k").is_between(*LEVELS["high"])).then(pl.lit("high"))
    e = e.when(pl.col("k").is_between(*LEVELS["mid"])).then(pl.lit("mid"))
    return e.when(pl.col("k").is_between(*LEVELS["low"])).then(pl.lit("low")).otherwise(None)


# --- figure style ---------------------------------------------------------------------------------------
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2})


def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=400, bbox_inches="tight")


print(f"D = {D} | {len(KS)} budgets | {len(RANKED_METHODS)} ranked configurations | lambda runs: {LAMBDA_IDS}")
for lvl, (lo, hi) in LEVELS.items():
    print(f"{lvl:>4} compression: k = {lo:,}-{hi:,}")

## 1. Data and candidate subgraph $G_D$

$G_D$ is the union of the forward $D$-hop neighbourhoods of the mapped concepts, with the root concept removed (notebook 1).

In [ ]:
id_to_label, G_d = graph_fct.get_combined_combined_subgraphs_and_id2label()
mapped_ids = sorted(pl.read_parquet(config.BasicConfig().mapped_path)["id"].unique().to_list())
label = lambda c: id_to_label.get(c, c)

rel_d = {d["relation"] for _, _, d in G_d.edges(data=True)}
print(f"|M| = {len(mapped_ids):,} mapped concepts ({sum(c in G_d for c in mapped_ids):,} in G_D)")
print(f"G_D: {G_d.number_of_nodes():,} nodes, {G_d.number_of_edges():,} edges, {len(rel_d)} relation types, "
      f"{nx.number_weakly_connected_components(G_d)} weakly connected components, DAG: {nx.is_directed_acyclic_graph(G_d)}")

In [ ]:
# full ontology vs G_D (experimental setup, appendix "absent relation types")
if SHOW_GRAPH_STATS:
    df_relations = pl.read_parquet(config.BasicConfig().relation_path)
    G_original = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
    rel_orig = set(df_relations["relation"].unique().to_list())
    print(f"Full ontology: {G_original.number_of_nodes():,} nodes, {G_original.number_of_edges():,} edges, "
          f"{len(rel_orig)} relation types, {nx.number_weakly_connected_components(G_original)} weakly connected "
          f"components, DAG: {nx.is_directed_acyclic_graph(G_original)}")
    print("Relation types absent from G_D:", sorted(rel_orig - rel_d))
    del G_original

## 2. Evaluation of every configuration at every budget

For each (method, $k$), the context trees of all $|M|$ concepts are built once, and `eval.evaluate` computes every metric from them (`tokenizer._worker_coverage_score` adds the semantic coverage).
* **Baselines and set-cover ablation:** the top $k$ of each ranking.
* **Random:** 20 draws per $k$, averaged after evaluation.
* **Ours:** the top $k$ of each $\lambda$ run. Every $k$ is a prefix of the greedy order.

A cached file is reused only if it already contains `br_weighted_PC` and `num_effective_token`.

In [ ]:
NEW_METRICS = {"br_weighted_PC", "num_effective_token"}

# built once and shared by every task: they depend on the graph only, not on the token set
A, node_to_idx = tokenizer.build_coverage_transition(G_d)     # semantic coverage at lambda = 1 (versioning only)
adj = tokenizer.build_out_adjacency(G_d)


def cached(path):
    # True if path holds results that already include the new metrics
    if RECOMPUTE or not os.path.exists(path):
        return False
    return NEW_METRICS <= set(pl.read_parquet_schema(path))


def run_tasks(tasks, path, label_cols):
    # evaluate (label, T) tasks in parallel; one row per task, saved to path
    if cached(path):
        print(f"cached: {path}")
        return pl.read_parquet(path)
    rows = []
    # spawn (the Windows default) on every OS; labels are not needed for the metrics (id_to_label=None)
    with mp.get_context("spawn").Pool(N_WORKERS, initializer=tokenizer._init_coverage_worker,
                                      initargs=(A, node_to_idx, adj, mapped_ids, D, None)) as pool:
        results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
        for lab, metrics in tqdm(results, total=len(tasks), desc=os.path.basename(path)):
            rows.append({**dict(zip(label_cols, lab)), **metrics})
    df = pl.DataFrame(rows).sort(label_cols)
    df.write_parquet(path)
    return df


def top_k_tasks(method, path):
    toks = pl.read_parquet(path)["token"].to_list()
    return [((method, k), toks[:k]) for k in KS]

In [ ]:
# baselines and set-cover ablation: top k of each ranking
tasks = [t for m in [ABLATION] + BASELINES for t in top_k_tasks(m, getattr(config.CandidateLists(), m))]
perf_base = run_tasks(tasks, OUT_DIR + "perf_baselines.parquet", ["method", "k"])

In [ ]:
# random: len(RND_ITERS) draws per budget
df_rnd = pl.read_parquet(config.CandidateLists().k_random_all_samples)
draws = {(int(k), int(it)): g["token"].to_list() for (k, it), g in df_rnd.group_by(["k", "iter"])}
tasks = [((RANDOM, k, it), draws[(k, it)]) for k in KS for it in RND_ITERS]
perf_rnd = run_tasks(tasks, OUT_DIR + "perf_random.parquet", ["method", "k", "iter"])

In [ ]:
# ours: every lambda run (incl. the fine grid 0.91-0.99), top-k prefixes of the greedy order
tasks = [t for l in LAMBDA_IDS for t in top_k_tasks(l, OURS_FILES[l])]
perf_ours = run_tasks(tasks, OUT_DIR + "perf_ours.parquet", ["method", "k"])

In [ ]:
perf_raw = (pl.concat([perf_base, perf_rnd, perf_ours], how="diagonal_relaxed")
              .with_columns(pl.col("iter").fill_null(-1))
              .rename(RENAME))
perf_raw.write_parquet(OUT_DIR + "performance_raw.parquet")
METRICS_ALL = [c for c in perf_raw.columns if c not in ("method", "k", "iter")]

# one row per (method, k): random averaged over its draws
perf = (perf_raw.group_by(["method", "k"]).agg([pl.col(m).mean() for m in METRICS_ALL])
                .with_columns((pl.col("num_effective_token") / pl.col("k")).alias("effective_token_share"),
                              pl.col("method").replace_strict(FAMILY, default=None).alias("family"),
                              pl.col("method").replace_strict(DISPLAY, default=None).alias("display"))
                .sort(["method", "k"]))
perf.write_parquet(OUT_DIR + "performance_all.parquet")
perf.write_csv(OUT_DIR + "performance_all.csv")
perf.select("method", "k", *CORE, "num_effective_token", "distance_score").head()

In [ ]:
# versioning: the current PC (distance_score) must equal the results of notebook 4
if os.path.exists(config.Results().perf_all):
    old = (pl.read_parquet(config.Results().perf_all)
             .select(pl.col("file_type").alias("method"), "k", pl.col("distance_score").alias("distance_score_nb4")))
    chk = (perf.join(old, on=["method", "k"], how="inner")
               .with_columns((pl.col("distance_score") - pl.col("distance_score_nb4")).abs().alias("diff")))
    det, rnd = chk.filter(pl.col("method") != RANDOM), chk.filter(pl.col("method") == RANDOM)
    print(f"current PC vs notebook 4: {det.height} (method, k) pairs, max |difference| = {det['diff'].max():.2e}")
    if rnd.height:   # random depends on the draws stored in k_random_all_samples.parquet
        print(f"random: {rnd.height} budgets, max |difference| = {rnd['diff'].max():.2e}")

## 3. Ranking of the configurations

1. At each $k$, rank the configurations on each core metric (1 = best).
2. Average the four ranks.
3. Average over the budgets of each compression level.

The fine grid $\lambda$ = 0.91–0.99 is not ranked. Section 5 uses it.

In [ ]:
ranked = (perf.filter(pl.col("method").is_in(RANKED_METHODS))
              .with_columns(level_expr().alias("level"))
              .filter(pl.col("level").is_not_null())
              .with_columns([pl.col(m).rank(method="average", descending=HIB[m]).over("k").alias(f"{m}_rank")
                             for m in CORE])
              .with_columns(pl.mean_horizontal([f"{m}_rank" for m in CORE]).alias("avg_rank")))
rank_table = (ranked.group_by(["level", "family", "method", "display"])
                    .agg([pl.col(f"{m}_rank").mean() for m in CORE] + [pl.col("avg_rank").mean()])
                    .sort(["level", "avg_rank"]))
rank_table.write_csv(OUT_DIR + "ranking.csv")

for lvl in LEVELS:
    d = rank_table.filter(pl.col("level") == lvl)
    best_ours = d.filter(pl.col("family") == "ours").row(0, named=True)
    best_other = d.filter(pl.col("family") != "ours").row(0, named=True)
    print(f"{lvl:>4}: best ours = {best_ours['display']} ({best_ours['avg_rank']:.2f}) | "
          f"best other = {best_other['display']} ({best_other['avg_rank']:.2f})")
rank_table

In [ ]:
# LaTeX rows of the appendix ranking table: method | rank per core metric | average rank
for lvl in LEVELS:
    print(f"% --- {lvl} compression")
    for r in rank_table.filter(pl.col("level") == lvl).iter_rows(named=True):
        cells = [r["display"]] + [f"{r[f'{m}_rank']:.1f}" for m in CORE] + [f"{r['avg_rank']:.2f}"]
        print(" & ".join(cells) + r" \\")

In [ ]:
# Figure: average rank by method family and compression level (ours coloured by lambda)
FAMILIES = [("random", "Random"), ("degree-based", "Degree"), ("centrality-based", "Centrality"),
            ("ours(ablation)", "Ablation"), ("ours", "Ours")]
BASE_COLOR = "#898781"
lam_cmap = LinearSegmentedColormap.from_list("lam_viridis", plt.cm.viridis(np.linspace(0.0, 0.8, 256)))
n_methods = len(RANKED_METHODS)

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.3), sharey=True)
x_pos = {fam: i for i, (fam, _) in enumerate(FAMILIES)}
for ax, (lvl, (lo, hi)) in zip(axes, LEVELS.items()):
    d = rank_table.filter(pl.col("level") == lvl)
    for fam, _ in FAMILIES[:-1]:
        sub = d.filter(pl.col("family") == fam)
        ax.scatter(np.full(sub.height, x_pos[fam]), sub["avg_rank"], s=30, color=BASE_COLOR,
                   edgecolor="white", linewidth=0.5, zorder=3, alpha=0.7)
    ours = d.filter(pl.col("family") == "ours").with_columns(pl.col("method").cast(pl.Float64).alias("lam"))
    ax.scatter(np.full(ours.height, x_pos["ours"]), ours["avg_rank"], s=34, c=ours["lam"], cmap=lam_cmap,
               vmin=0, vmax=1, edgecolor="white", linewidth=0.8, zorder=4, alpha=0.7)
    best = ours.sort("avg_rank").row(0, named=True)
    ax.scatter([x_pos["ours"]], [best["avg_rank"]], s=110, facecolor="none", edgecolor=INK, linewidth=1.0, zorder=5)
    ax.annotate(f"$\\lambda$ = {best['lam']:.1f}", (x_pos["ours"], best["avg_rank"]), xytext=(-9, 0),
                textcoords="offset points", ha="right", va="center", fontsize=7, color=INK)
    ax.set_title(f"{lvl.capitalize()} compression\n$k$ = {lo:,}–{hi:,}", fontsize=8, color=INK)
    ax.set_xticks(range(len(FAMILIES)))
    ax.set_xticklabels([lab for _, lab in FAMILIES], rotation=40, ha="right", fontsize=7)
    ax.set_xlim(-0.6, len(FAMILIES) - 0.05)
    ax.grid(axis="y", color=GRID, lw=0.5)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylim(n_methods + 0.5, 0.5)
axes[0].set_yticks(sorted({1, 5, 10, 15, n_methods} & set(range(1, n_methods + 1))))
axes[0].set_ylabel(f"Average rank over {len(CORE)} core metrics (1 = best)")
sm = plt.cm.ScalarMappable(cmap=lam_cmap, norm=plt.Normalize(0, 1))
cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.06)
cbar.set_label("Distance decay $\\lambda$ (ours)", color=INK2)
cbar.outline.set_visible(False)
save_fig(fig, "rank_by_family")

## 4. Effect of the token budget $k$

Main text: the four core metrics. Appendix: the same figure with random selection and the uncovered branch rate.

In [ ]:
SERIES = {
    LAM_MAIN:                ("Ours ($\\lambda{=}" + LAM_MAIN + "$)", "#2a78d6", 2.2, "-"),
    "personalized_pagerank": ("Pers. PageRank",                    "#1baf7a", 1.4, "-"),
    "highest_degree_dist_1": ("Highest degree ($d \\leq 1$)",      "#eda100", 1.4, "-"),
    ABLATION:                ("Set cover (ablation)",              "#898781", 1.3, (0, (5, 2))),
    RANDOM:                  ("Random",                            "#898781", 1.3, (0, (1, 1.5))),
}
LEVEL_EDGES = [(LEVELS["high"][1] + LEVELS["mid"][0]) / 2, (LEVELS["mid"][1] + LEVELS["low"][0]) / 2]
LEVEL_LABELS = [((lo + hi) / 2, lvl.capitalize()) for lvl, (lo, hi) in LEVELS.items()]


def plot_metrics_vs_k(df_k, metrics, series, ncols=3, panel_size=(2.5, 2)):
    n_panels = len(metrics) + 1                                   # +1 slot for the legend
    nrows = -(-n_panels // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_size[0] * ncols, panel_size[1] * nrows),
                             sharex=True, constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    for ax, metric in zip(axes, metrics):
        for x in LEVEL_EDGES:
            ax.axvline(x, color=GRID, lw=0.8, zorder=0)
        for meth, (lab, color, lw, ls) in reversed(list(series.items())):
            sub = df_k.filter(pl.col("method") == meth).sort("k")
            if sub.height:
                ax.plot(sub["k"], sub[metric], color=color, lw=lw, ls=ls, label=lab,
                        zorder=3 if meth == LAM_MAIN else 2, solid_capstyle="round")
        ax.set_title(METRIC_TITLE.get(metric, metric), fontsize=8, color=INK, pad=11)
        ax.grid(axis="y", color=GRID, lw=0.5)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for x, lab in LEVEL_LABELS:
            ax.text(x, 1.01, lab, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=6, color=MUTED)
        ax.xaxis.set_major_locator(MultipleLocator(5000))
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:g}k"))
        ax.tick_params(labelbottom=True)
        ax.set_xlabel("Token budget $k$", color=INK2)
    for ax in axes[len(metrics):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    order = [labels.index(series[m][0]) for m in series if series[m][0] in labels]
    axes[len(metrics)].legend([handles[i] for i in order], [labels[i] for i in order], loc="center",
                              frameon=False, fontsize=7.5, labelcolor=INK2, handlelength=2.6)
    return fig


main_series = {m: s for m, s in SERIES.items() if m != RANDOM}
save_fig(plot_metrics_vs_k(perf, CORE, main_series), "metrics_vs_k")
save_fig(plot_metrics_vs_k(perf, CORE + ["uncovered_branch_rate"], SERIES), "metrics_vs_k_all")

In [ ]:
# budgets at which each method first reaches a milestone (quotable in the text)
milestones = (perf.filter(pl.col("method").is_in(RANKED_METHODS))
                  .group_by(["method", "display"])
                  .agg(pl.col("k").filter(pl.col("unknown_rate") == 0).min().alias("min_k_unknown_0"),
                       pl.col("k").filter(pl.col("unique_rate") >= 0.9).min().alias("min_k_unique_90"),
                       pl.col("br_weighted_PC").max().alias("max_br_weighted_PC"))
                  .sort("min_k_unknown_0", nulls_last=True))
milestones.write_csv(OUT_DIR + "milestones_vs_k.csv")
milestones

## 5. Effect of the distance decay $\lambda$

Our method only. Each value is the mean over the budgets of a compression level.
* The figure leaves out $\lambda$ = 0. The table keeps it.
* The fine grid 0.91–0.99 shows what happens just below $\lambda$ = 1 (lines only; markers at steps of 0.1).

In [ ]:
LEVEL_STYLE = {"high": ("#6da7ec", "o"), "mid": ("#2a78d6", "s"), "low": ("#104281", "^")}
METRICS_LAMBDA = CORE + ["uncovered_branch_rate"]

df_lam = (perf.filter(pl.col("family") == "ours")
              .with_columns(pl.col("method").cast(pl.Float64).alias("lam"), level_expr().alias("level"))
              .filter(pl.col("level").is_not_null())
              .group_by(["level", "lam"]).agg([pl.col(m).mean() for m in METRICS_ALL + ["effective_token_share"]])
              .sort(["level", "lam"]))
df_lam.write_csv(OUT_DIR + "effect_of_lambda.csv")


def plot_metrics_vs_lambda(df_lam, metrics, ncols=3, panel_size=(2.5, 2)):
    nrows = -(-len(metrics) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_size[0] * ncols, panel_size[1] * nrows),
                             sharex=True, constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    for ax, metric in zip(axes, metrics):
        ax.axvline(float(LAM_MAIN), color=GRID, lw=0.8, zorder=0)
        ax.text(float(LAM_MAIN), 1.01, f"$\\lambda{{=}}{LAM_MAIN}$", transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=6, color=MUTED)
        for lvl, (color, mk) in LEVEL_STYLE.items():
            sub = df_lam.filter((pl.col("level") == lvl) & (pl.col("lam") > 0)).sort("lam")
            coarse = [i for i, l in enumerate(sub["lam"].to_list()) if abs(10 * l - round(10 * l)) < 1e-9]
            ax.plot(sub["lam"], sub[metric], color=color, lw=1.6, marker=mk, markersize=2.5, markevery=coarse,
                    markeredgecolor="white", markeredgewidth=0.5, label=f"{lvl.capitalize()} compression")
        ax.set_title(METRIC_TITLE.get(metric, metric), fontsize=8, color=INK, pad=11)
        ax.grid(axis="y", color=GRID, lw=0.5)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        ax.xaxis.set_major_locator(MultipleLocator(0.2))
        ax.set_xlim(0.05, 1.03)
        ax.tick_params(labelbottom=True)
        ax.set_xlabel("Distance decay $\\lambda$", color=INK2)
    for ax in axes[len(metrics):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0), bbox_transform=fig.transFigure,
               ncol=3, frameon=False, fontsize=7.5, labelcolor=INK2,
               title=f"Mean over the {n_lvl} budgets of each level", title_fontsize=7, handlelength=2.6)
    return fig


save_fig(plot_metrics_vs_lambda(df_lam, METRICS_LAMBDA), "metrics_vs_lambda")

# LaTeX rows of the appendix lambda table: coarse lambdas incl. 0, one block per level
print("% lambda & " + " & ".join(METRICS_LAMBDA))
for lvl in LEVELS:
    print(f"% --- {lvl} compression")
    for r in df_lam.filter((pl.col("level") == lvl) & pl.col("lam").is_in([float(l) for l in LAMBDA_COARSE])).iter_rows(named=True):
        print(f"{r['lam']:.1f} & " + " & ".join(f"{r[m]:.3f}" for m in METRICS_LAMBDA) + r" \\")

## 6. Per-concept distribution at fixed $k$

Our method, one budget per compression level (`KS_DIST`) and four values of $\lambda$ (`LAMS_DIST`).
* (a) branch-weighted PC, (b) tokens per concept.
* Box: interquartile range. Line: median. Whiskers: P10–P90. Dot: mean.

In [ ]:
path_dist = OUT_DIR + "per_concept_dist.parquet"
if RECOMPUTE or not os.path.exists(path_dist):
    frames = []
    for lam in tqdm(LAMS_DIST, desc="per-concept tables"):
        toks = pl.read_parquet(OURS_FILES[lam])["token"].to_list()
        for k in KS_DIST:
            _, pc = fe.evaluate_token_set(mapped_ids, adj, toks[:k], D, return_per_concept=True)
            frames.append(pc.select("concept", "br_weighted_PC", "br_completeness", "br_proximity", "fidelity",
                                    "n_tokens", "unknown", "has_uncovered_branch")
                            .with_columns(pl.lit(k).alias("k"), pl.lit(float(lam)).alias("lam")))
    pl.concat(frames).write_parquet(path_dist)
dist = pl.read_parquet(path_dist)

# checks: (1) the per-concept table gives the means of Section 2;
# (2) the vectorised br_weighted_PC equals its recursive definition (eval.br_weighted_pc_concept)
lam0, k0 = LAM_MAIN, KS_DIST[0]
sub = dist.filter((pl.col("lam") == float(lam0)) & (pl.col("k") == k0))
r0 = perf.filter((pl.col("method") == lam0) & (pl.col("k") == k0))
if r0.height:
    assert abs(sub["br_weighted_PC"].mean() - r0["br_weighted_PC"][0]) < 1e-9
T0 = set(pl.read_parquet(OURS_FILES[lam0])["token"].to_list()[:k0])
sample = sub.sample(n=min(200, sub.height), seed=0)
dev = max(abs(ev.br_weighted_pc_concept(c, adj, T0, D) - v) for c, v in zip(sample["concept"], sample["br_weighted_PC"]))
print(f"vectorised vs recursive br_weighted_PC on {sample.height} concepts: max |difference| = {dev:.1e}")

q = lambda c, p: pl.col(c).quantile(p, "nearest")
dist_summary = (dist.group_by(["k", "lam"])
                    .agg(*[e for c in ["br_weighted_PC", "n_tokens"]
                           for e in (q(c, .10).alias(f"{c}_p10"), q(c, .25).alias(f"{c}_p25"), q(c, .5).alias(f"{c}_median"),
                                     q(c, .75).alias(f"{c}_p75"), q(c, .90).alias(f"{c}_p90"), pl.col(c).mean().alias(f"{c}_mean"))])
                    .sort(["k", "lam"]))
dist_summary.write_csv(OUT_DIR + "per_concept_distribution.csv")
dist_summary

In [ ]:
_VIR = plt.cm.viridis
LAM_COLOR = {0.3: _VIR(0.05), 0.6: _VIR(0.36), 0.9: _VIR(0.62), 1.0: MUTED}   # lambda = 1: no decay
LAM_COLOR = {float(l): LAM_COLOR.get(float(l), _VIR(float(l))) for l in LAMS_DIST}
LEVEL_OF_K = {k: lvl for k in KS_DIST for lvl, (lo, hi) in LEVELS.items() if lo <= k <= hi}


def boxes(ax, d, col, width):
    lams = [float(l) for l in LAMS_DIST]
    for i, k in enumerate(KS_DIST):
        for j, lam in enumerate(lams):
            v = d.filter((pl.col("k") == k) & (pl.col("lam") == lam))[col].to_numpy()
            if len(v) == 0:
                continue
            x = i + (j - (len(lams) - 1) / 2) * width
            p10, p25, p50, p75, p90 = np.percentile(v, [10, 25, 50, 75, 90])
            c = LAM_COLOR[lam]
            ax.add_patch(plt.Rectangle((x - width * 0.42, p25), width * 0.84, max(p75 - p25, 1e-9),
                                       facecolor=c, edgecolor=c, alpha=0.35, lw=0, zorder=2))
            ax.add_patch(plt.Rectangle((x - width * 0.42, p25), width * 0.84, max(p75 - p25, 1e-9),
                                       fill=False, edgecolor=c, lw=0.9, zorder=3))
            ax.plot([x, x], [p10, p25], color=c, lw=0.9, zorder=2)
            ax.plot([x, x], [p75, p90], color=c, lw=0.9, zorder=2)
            ax.plot([x - width * 0.42, x + width * 0.42], [p50, p50], color=c, lw=2.0, zorder=4, solid_capstyle="butt")
            ax.plot(x, v.mean(), marker="o", ms=2.6, color="white", markeredgecolor=c, markeredgewidth=0.8, zorder=5)


fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), constrained_layout=True)
w = 0.2
boxes(axes[0], dist, "br_weighted_PC", w)
axes[0].set_title("(a) Branch-weighted PC $\\uparrow$", fontsize=7.5, color=INK, pad=5)
low = float(dist_summary["br_weighted_PC_p10"].min())
axes[0].set_ylim(max(0.0, low - 0.05), 1.03)
boxes(axes[1], dist, "n_tokens", w)
axes[1].set_title("(b) Tokens per concept $\\downarrow$", fontsize=7.5, color=INK, pad=5)
axes[1].set_ylim(0, None)
for ax in axes:
    ax.set_xlim(-0.6, len(KS_DIST) - 0.4)
    ax.set_xticks(range(len(KS_DIST)))
    ax.set_xticklabels([f"{k:,}\n({LEVEL_OF_K.get(k, '')})" for k in KS_DIST])
    ax.set_xlabel("Token budget $k$ (compression level)", fontsize=7, color=INK2)
    ax.grid(axis="y", color=GRID, lw=0.5)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=2.5, width=0.6, labelsize=6.8)
handles = [Patch(facecolor=LAM_COLOR[float(l)], edgecolor=LAM_COLOR[float(l)], alpha=0.8,
                 label=(f"$\\lambda = {l}$" if float(l) < 1 else "$\\lambda = 1$ (no decay)")) for l in LAMS_DIST]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), bbox_transform=fig.transFigure,
           ncol=len(LAMS_DIST), frameon=False, fontsize=7.2)
save_fig(fig, "dist_k_lambda")

## 7. Effectively used tokens (informational)

`num_effective_token` counts the selected tokens that at least one mapped concept uses. A token is unused when every concept that could reach it stops earlier at another token. This metric is not ranked.
* `share_*`: mean of `num_effective_token / k` over the budgets of each level.
* `used_k*`: `num_effective_token` at the budgets of `KS_DIST`.

In [ ]:
eff_share = (perf.filter(pl.col("method").is_in(RANKED_METHODS))
                 .with_columns(level_expr().alias("level"))
                 .filter(pl.col("level").is_not_null())
                 .group_by(["method", "display", "family", "level"])
                 .agg(pl.col("effective_token_share").mean())
                 .pivot(on="level", index=["method", "display", "family"], values="effective_token_share")
                 .rename({lvl: f"share_{lvl}" for lvl in LEVELS})
                 .select("method", "display", "family", *[f"share_{lvl}" for lvl in LEVELS]))
eff_at_k = (perf.filter(pl.col("method").is_in(RANKED_METHODS) & pl.col("k").is_in(KS_DIST))
                .select("method", pl.col("k").cast(pl.String),
                        pl.col("num_effective_token").round(0).cast(pl.Int64))       # random: mean of the draws
                .pivot(on="k", index="method", values="num_effective_token")
                .rename({str(k): f"used_k{k}" for k in KS_DIST if k in KS}))
eff = eff_share.join(eff_at_k, on="method", how="left").sort("share_high", descending=True)
eff.write_csv(OUT_DIR + "effective_tokens.csv")
with pl.Config(tbl_rows=30, tbl_cols=12, fmt_str_lengths=40):
    display(eff.drop("method"))

## 8. Correlation between metrics, by method family

Spearman correlation over all (method, $k$) rows of a family. Do the core metrics measure different things?

In [ ]:
CORR_COLS = CORE + ["uncovered_branch_rate", "effective_token_share"]
fam_list = ["degree-based", "centrality-based", "random", "ours", "ours(ablation)"]
fig, axes = plt.subplots(2, 3, figsize=(9.0, 6.4), constrained_layout=True)
axes = axes.flatten()
axes[-1].axis("off")
corr_rows = []
for ax, fam in zip(axes, fam_list):
    sub = perf.filter(pl.col("family") == fam).select(CORR_COLS)
    ranks = np.column_stack([sub[c].rank("average").to_numpy() for c in CORR_COLS])
    with np.errstate(invalid="ignore", divide="ignore"):
        corr = np.corrcoef(ranks, rowvar=False)
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(CORR_COLS)))
    ax.set_yticks(range(len(CORR_COLS)))
    ax.set_xticklabels([c.replace("_", " ") for c in CORR_COLS], rotation=60, ha="right", fontsize=6)
    ax.set_yticklabels([c.replace("_", " ") for c in CORR_COLS], fontsize=6)
    ax.set_title(fam, fontsize=8)
    for i in range(len(CORR_COLS)):
        for j in range(len(CORR_COLS)):
            v = corr[i, j]
            ax.text(j, i, "" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=5,
                    color="white" if abs(np.nan_to_num(v)) > 0.6 else "black")
            corr_rows.append({"family": fam, "metric_1": CORR_COLS[i], "metric_2": CORR_COLS[j], "spearman": v})
fig.colorbar(im, ax=axes[:len(fam_list)], shrink=0.6, label="Spearman correlation")
save_fig(fig, "metric_correlations")
pl.DataFrame(corr_rows).write_csv(OUT_DIR + "metric_correlations.csv")

## 9. Key numbers for the text

Collected in `OUT_DIR/key_numbers.json`. Keep few numbers in the prose and point to the tables and figures.

In [ ]:
def row(df, **eq):
    d = df
    for k_, v in eq.items():
        d = d.filter(pl.col(k_) == v)
    return d.row(0, named=True) if d.height else None


KEY_COLS = CORE + ["uncovered_branch_rate", "num_effective_token", "effective_token_share"]
key = {
    "|M|": len(mapped_ids),
    "G_D nodes / edges": (G_d.number_of_nodes(), G_d.number_of_edges()),
    "compression levels (k ranges)": LEVELS,
    "best configuration per level": {lvl: rank_table.filter(pl.col("level") == lvl).row(0, named=True)["display"]
                                     for lvl in LEVELS},
    f"average rank of ours (lambda = {LAM_MAIN}) per level": {
        lvl: row(rank_table, level=lvl, method=LAM_MAIN)["avg_rank"] for lvl in LEVELS},
    f"ours (lambda = {LAM_MAIN}) at k": {k: {m: row(perf, method=LAM_MAIN, k=k)[m] for m in KEY_COLS}
                                          for k in KS_DIST if row(perf, method=LAM_MAIN, k=k)},
    "personalized PageRank at k": {k: {m: row(perf, method="personalized_pagerank", k=k)[m] for m in KEY_COLS}
                                   for k in KS_DIST if row(perf, method="personalized_pagerank", k=k)},
}
with open(OUT_DIR + "key_numbers.json", "w") as f:
    json.dump(key, f, indent=2, default=float)
print(json.dumps(key, indent=2, default=float))